For discussion, refer to:   
https://www.kaggle.com/competitions/vesuvius-challenge-surface-detection/discussion/667135

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Helpers: coords + grid_sample


def zyx_to_grid_norm(x_zyx: torch.Tensor, vol_shape_zyx):
    """
    x_zyx: (N,3) in voxel coords, z in [0,Z-1], y in [0,Y-1], x in [0,X-1]
    returns: (1,N,1,1,3) normalized coords for grid_sample in order (x,y,z)
    """
    Z, Y, X = vol_shape_zyx
    z = x_zyx[:, 0]
    y = x_zyx[:, 1]
    x = x_zyx[:, 2]

    # normalize to [-1,1]
    # align_corners=True -> -1 maps to 0, +1 maps to size-1
    x_n = 2.0 * x / (X - 1) - 1.0
    y_n = 2.0 * y / (Y - 1) - 1.0
    z_n = 2.0 * z / (Z - 1) - 1.0

    grid = torch.stack([x_n, y_n, z_n], dim=-1)  # (N,3) in (x,y,z)
    grid = grid.view(1, -1, 1, 1, 3)
    return grid

def sample_velocity(u_grid_zyx: torch.Tensor, x_zyx: torch.Tensor, vol_shape_zyx):
    """
    u_grid_zyx: (1,3,Zg,Yg,Xg) velocity in ZYX component order (dz,dy,dx) at grid resolution.
    x_zyx: (N,3) points in voxel coords (ZYX) at volume resolution.
    returns: (N,3) velocity at points in voxel units per unit time.
    """
    # grid_sample expects input shape (N,C,D,H,W), grid (N, Dout, Hout, Wout, 3)
    # We'll treat points as (Dout=N, Hout=1, Wout=1)
    # Need normalized coords w.r.t. u_grid resolution, not volume resolution.
    Zg, Yg, Xg = u_grid_zyx.shape[-3], u_grid_zyx.shape[-2], u_grid_zyx.shape[-1]

    # Convert x_zyx (volume voxel coords) to u-grid voxel coords
    Z, Y, X = vol_shape_zyx
    z = x_zyx[:, 0] * (Zg - 1) / max(Z - 1, 1)
    y = x_zyx[:, 1] * (Yg - 1) / max(Y - 1, 1)
    x = x_zyx[:, 2] * (Xg - 1) / max(X - 1, 1)
    xg_zyx = torch.stack([z, y, x], dim=-1)

    # normalized coords for u-grid
    grid = zyx_to_grid_norm(xg_zyx, (Zg, Yg, Xg))  # (1,N,1,1,3) in (x,y,z) normalized

    # sample: output (1,3,N,1,1)
    v = F.grid_sample(
        u_grid_zyx, grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True
    )
    v = v.view(3, -1).transpose(0, 1).contiguous()  # (N,3), still in (dz,dy,dx)
    return v

def euler_integrate(u_grid_zyx: torch.Tensor, x0_zyx: torch.Tensor, vol_shape_zyx,
                    n_steps=16, dt=1.0/16.0):
    """
    Integrate dx/dt = u(x) with Euler steps.
    x0_zyx: (N,3) in voxel coords
    """
    x = x0_zyx
    for _ in range(n_steps):
        v = sample_velocity(u_grid_zyx, x, vol_shape_zyx)  # (N,3) in voxel units
        x = x + dt * v
    return x

In [ ]:
#self = canoical

class CanoicalFitter (nn.Module):
    def __init__(self, vol_shape_zyx, grid_shape_zyx=(40,40,40),
                 n_steps=16):
        super().__init__()
        self.vol_shape_zyx = tuple(vol_shape_zyx)
        self.n_steps = n_steps

        Zg, Yg, Xg = grid_shape_zyx
        # Learnable velocity field on coarse grid: (1,3,Zg,Yg,Xg) in (dz,dy,dx)
        self.u = nn.Parameter(torch.zeros(1, 3, Zg, Yg, Xg))

        #---
        # todo ... initialisae surface to median of data ....

        
        # Learnable sheet gap (positive)
        self.log_g = nn.Parameter(torch.tensor(0.0))

    @torch.no_grad()
    def set_canonical_frame(self, mu_zyx, n_zyx):
        mu_zyx = torch.as_tensor(mu_zyx, dtype=torch.float32, device=self.mu_zyx.device)
        n_zyx  = torch.as_tensor(n_zyx, dtype=torch.float32, device=self.n_zyx.device)
        n_zyx  = n_zyx / (n_zyx.norm() + 1e-8)
        self.mu_zyx.copy_(mu_zyx)
        self.n_zyx.copy_(n_zyx)

    def V_to_S(self, xV_zyx):
        """
        Approx inverse map by integrating with -u (paper trick).
        """
        xS_zyx = euler_integrate(-self.u, xV_zyx, self.vol_shape_zyx, n_steps=self.n_steps, dt=1.0/self.n_steps)
        return xS_zyx

    def sheet_distance(self, xS_zyx):
        """
        Distance of canonical-mapped points to the nearest of two parallel sheets
        along canonical normal axis n (stored).
        """
        g = torch.exp(self.log_g) + 1e-6  # gap in voxel units along n
        # canonical coordinate w = n·(x - mu)
        w = ((xS_zyx - self.mu_zyx) * self.n_zyx).sum(dim=-1)  # (N,)
        d1 = (w + 0.5 * g).abs()
        d2 = (w - 0.5 * g).abs()

        # softmin is smoother than min()
        tau = 0.25
        d = -tau * torch.log(torch.exp(-d1/tau) + torch.exp(-d2/tau))
        return d, w, g

    def smoothness_loss(self):
        """
        Simple ||∇u||^2 on the coarse grid.
        """
        u = self.u
        dz = u[:,:,1:,:,:] - u[:,:,:-1,:,:]
        dy = u[:,:,:,1:,:] - u[:,:,:,:-1,:]
        dx = u[:,:,:,:,1:] - u[:,:,:,:,:-1]
        return (dz.pow(2).mean() + dy.pow(2).mean() + dx.pow(2).mean())


def fit_flow(
    point_zyx,
    canoical
):

    #canoical.parameters is the surface 
    model = canoical
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    P = point_zyx

    for it in range(iters):
        idx = torch.randint(0, P.shape[0], (batch_size,), device=device)
        xV = P[idx]  # (B,3)

        xS = model.V_to_S(xV)
        d, w, g = model.distance_loss(xS)

        loss_data   = d.mean()
        loss_smooth = model.smoothness_loss()
        loss_small  = model.u.pow(2).mean()
        loss_gap    = 1.0 / (g + 1e-3)  # prevents collapse to g->0

        loss = 1.0*loss_data + 0.05*loss_smooth + 0.001*loss_small + 0.01*loss_gap

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()



In [3]:
import numpy as np

input_3d = np.load("/kaggle/input/blabla/kaggle/working/1033784946_pred.npy")
print(input_3d.shape)

(320, 320, 320)


In [9]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import binary_dilation

# -----------------------------
# Helper functions
# -----------------------------

def zyx_to_grid_norm(x_zyx: torch.Tensor, vol_shape_zyx):
    Z, Y, X = vol_shape_zyx
    z = x_zyx[:, 0]
    y = x_zyx[:, 1]
    x = x_zyx[:, 2]

    x_n = 2.0 * x / (X - 1) - 1.0
    y_n = 2.0 * y / (Y - 1) - 1.0
    z_n = 2.0 * z / (Z - 1) - 1.0

    grid = torch.stack([x_n, y_n, z_n], dim=-1)
    grid = grid.view(1, -1, 1, 1, 3)
    return grid


def sample_velocity(u_grid_zyx, x_zyx, vol_shape_zyx):

    Zg, Yg, Xg = u_grid_zyx.shape[-3:]

    Z, Y, X = vol_shape_zyx
    z = x_zyx[:, 0] * (Zg - 1) / max(Z - 1, 1)
    y = x_zyx[:, 1] * (Yg - 1) / max(Y - 1, 1)
    x = x_zyx[:, 2] * (Xg - 1) / max(X - 1, 1)

    xg_zyx = torch.stack([z, y, x], dim=-1)
    grid = zyx_to_grid_norm(xg_zyx, (Zg, Yg, Xg))

    v = F.grid_sample(
        u_grid_zyx,
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True
    )

    v = v.view(3, -1).T
    return v


def euler_integrate(u_grid, x0, vol_shape, n_steps=16):
    dt = 1.0 / n_steps
    x = x0
    for _ in range(n_steps):
        v = sample_velocity(u_grid, x, vol_shape)
        x = x + dt * v
    return x


# -----------------------------
# Canonical Fitter
# -----------------------------

class CanoicalFitter(nn.Module):

    def __init__(self, vol_shape_zyx, grid_shape_zyx=(32,32,32), n_steps=16):
        super().__init__()

        self.vol_shape_zyx = vol_shape_zyx
        self.n_steps = n_steps

        Zg,Yg,Xg = grid_shape_zyx
        self.u = nn.Parameter(torch.zeros(1,3,Zg,Yg,Xg))

        self.log_g = nn.Parameter(torch.tensor(0.0))

        self.register_buffer("mu_zyx", torch.zeros(3))
        self.register_buffer("n_zyx", torch.tensor([1.0,0.0,0.0]))

    @torch.no_grad()
    def set_canonical_frame(self, mu, n):
        n = n / (n.norm() + 1e-8)
        self.mu_zyx.copy_(mu)
        self.n_zyx.copy_(n)

    def V_to_S(self, x):
        return euler_integrate(-self.u, x, self.vol_shape_zyx, self.n_steps)

    def sheet_distance(self, xS):

        g = torch.exp(self.log_g) + 1e-6

        w = ((xS - self.mu_zyx) * self.n_zyx).sum(-1)

        d1 = (w + 0.5*g).abs()
        d2 = (w - 0.5*g).abs()

        tau = 0.25
        # d = -tau * torch.log(torch.exp(-d1/tau) + torch.exp(-d2/tau))

        stacked = torch.stack([-d1/tau, -d2/tau], dim=0)
        d = -tau * torch.logsumexp(stacked, dim=0)


        return d, w, g

    def smoothness_loss(self):

        u = self.u

        dz = u[:,:,1:,:,:] - u[:,:,:-1,:,:]
        dy = u[:,:,:,1:,:] - u[:,:,:,:-1,:]
        dx = u[:,:,:,:,1:] - u[:,:,:,:,:-1]

        return dz.pow(2).mean() + dy.pow(2).mean() + dx.pow(2).mean()


# -----------------------------
# Training
# -----------------------------

def fit_flow(points, model, device):

    lr = 1e-3
    iters = 1500
    batch = 20000

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    P = points

    for i in range(iters):

        idx = torch.randint(0, P.shape[0], (batch,), device=device)
        xV = P[idx]

        xS = model.V_to_S(xV)
        d, w, g = model.sheet_distance(xS)

        # g = torch.clamp(torch.exp(np.log(g)), min=0.5, max=10.0)
        
        loss = (
            d.mean()
            + 0.05 * model.smoothness_loss()
            + 0.001 * model.u.pow(2).mean()
            + 0.01 * (1.0/(g+1e-3))
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

        if i % 100 == 0:
            print(i, loss.item(), g.item())


# -----------------------------
# MAIN PIPELINE
# -----------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load mask
input_3d = np.load("/kaggle/input/blabla/kaggle/working/1033784946_pred.npy")

# Dilate mask
mask = binary_dilation(input_3d.astype(bool), iterations=2)

# Extract points
points = torch.from_numpy(np.stack(np.where(mask),1)).float().to(device)

# Create model
model = CanoicalFitter((320,320,320)).to(device)

# Initialize canonical frame using PCA
mu = points.mean(0)
X = points - mu
cov = X.T @ X
eigvals, eigvecs = torch.linalg.eigh(cov)
n = eigvecs[:,0]

model.set_canonical_frame(mu, n)

# Train warp
fit_flow(points, model, device)

# -----------------------------
# Refinement
# -----------------------------

with torch.no_grad():

    xS = model.V_to_S(points)
    d,_,_ = model.sheet_distance(xS)

    good = d < 1.5
    refined_pts = points[good].long().cpu().numpy()

refined_mask = np.zeros_like(input_3d)
refined_mask[
    refined_pts[:,0],
    refined_pts[:,1],
    refined_pts[:,2]
] = 1

np.save("refined_mask.npy", refined_mask)


0 69.33422088623047 1.0000009536743164
100 69.26681518554688 1.1072036027908325
200 68.31780242919922 1.2323356866836548
300 68.588623046875 1.3790334463119507
400 67.74468994140625 1.5515873432159424
500 68.31446838378906 1.7552825212478638
600 68.04187774658203 1.9965311288833618
700 67.88656616210938 2.283389091491699
800 66.92611694335938 2.6259779930114746
900 66.95856475830078 3.0362424850463867
1000 66.83975982666016 3.529184341430664
1100 66.14786529541016 4.1233696937561035
1200 66.18399810791016 4.84133768081665
1300 65.21282196044922 5.710658550262451
1400 64.39604949951172 6.76494836807251


In [10]:
#Visualzie
%matplotlib inline
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def compare_3d_volume(
  vol1,
  vol2,
  axis=0,
  cmap="viridis"
):
  # vol1 = tifffile.imread(tiff_path_1)
  # vol2 = tifffile.imread(tiff_path_2)

  assert vol1.shape == vol2.shape
  assert axis in (0, 1, 2)

  vmin = min(vol1.min(), vol2.min())
  vmax = max(vol1.max(), vol2.max())

  def get_slice(vol, idx):
    if axis == 0:
      return vol[idx]
    elif axis == 1:
      return vol[:, idx, :]
    else:
      return vol[:, :, idx]

  slider = widgets.IntSlider(
    min=0,
    max=vol1.shape[axis] - 1,
    step=1,
    value=vol1.shape[axis] // 2,
    description="Slice",
    continuous_update=False
  )

  out = widgets.Output()

  def update(change=None):
    idx = slider.value
    with out:
      out.clear_output(wait=True)
      fig, axes = plt.subplots(1, 2, figsize=(12, 6))

      # REMOVE vmin=vmin, vmax=vmax to let matplotlib auto-scale
      im1 = axes[0].imshow(get_slice(vol1, idx), cmap=cmap)
      axes[0].set_title(f"Vol 1 (Auto-scaled) | Slice {idx}")
      plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
      axes[0].axis("off")

      im2 = axes[1].imshow(get_slice(vol2, idx), cmap=cmap)
      axes[1].set_title(f"Vol 2 (Auto-scaled) | Slice {idx}")
      plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
      axes[1].axis("off")

      plt.show()

      plt.show()

  slider.observe(update, names="value")

  display(widgets.VBox([slider, out]))
  update()

In [11]:


compare_3d_volume(input_3d, refined_mask)